# Deep Research with Clarifying Questions + Pushover Notifications
### Samuel Kalu, Team Euclid, Week 2


## Installation

In [ ]:
!pip install openai-agents openai gradio python-dotenv pydantic requests nest-asyncio --quiet

In [ ]:
import gradio as gr
from pydantic import BaseModel, Field
from agents import Agent, trace, Runner, WebSearchTool
from dotenv import load_dotenv
from IPython.display import display, Markdown
import asyncio
import nest_asyncio
from agents.model_settings import ModelSettings
import os
import requests
from typing import Dict

nest_asyncio.apply()
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
PUSHOVER_USER = os.getenv('PUSHOVER_USER')
PUSHOVER_TOKEN = os.getenv('PUSHOVER_TOKEN')

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found!")
print(f"✓ OpenAI API Key loaded")

SEND_NOTIFICATION = bool(PUSHOVER_USER and PUSHOVER_TOKEN)
if SEND_NOTIFICATION:
    print(f"✓ Pushover configured")
else:
    print("⚠ Pushover not configured - notifications disabled")

## Structured Outputs

In [ ]:
class Evaluation(BaseModel):
    is_satisfactory: bool
    feedback: str
    score: float

class WebSearchItem(BaseModel):
    reason: str
    query: str

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]

class ReportData(BaseModel):
    short_summary: str
    markdown_report: str
    follow_up_questions: list[str]

## Agents

In [ ]:
clarifying_agent = Agent(
    name="ClarifyingAgent",
    instructions="Ask 3 clarifying questions to understand the user's research needs. Format them nicely with markdown.",
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.7, max_tokens=350)
)

planner_agent = Agent(
    name="PlannerAgent",
    instructions="Plan 3 targeted web searches to answer the query.",
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
    model_settings=ModelSettings(temperature=0.3, max_tokens=500)
)

search_agent = Agent(
    name="SearchAgent",
    instructions="Search the web and produce 2-3 paragraph summaries.",
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required", temperature=0.3, max_tokens=400)
)

writer_agent = Agent(
    name="WriterAgent",
    instructions="Write a comprehensive report (1000+ words) in markdown format.",
    model="gpt-4o",
    output_type=ReportData,
    model_settings=ModelSettings(temperature=0.5, max_tokens=2000)
)

evaluator_agent = Agent(
    name="EvaluatorAgent",
    instructions="Evaluate if answers are sufficient for research. Be lenient.",
    model="gpt-4o-mini",
    output_type=Evaluation,
    model_settings=ModelSettings(temperature=0.2, max_tokens=400)
)

## Pushover Notification

In [ ]:
def send_notification(title: str, message: str) -> Dict:
    if not SEND_NOTIFICATION:
        return {"status": "disabled"}
    try:
        url = "https://api.pushover.net/1/messages.json"
        data = {"token": PUSHOVER_TOKEN, "user": PUSHOVER_USER, "title": title, "message": message[:1024]}
        response = requests.post(url, data=data, timeout=10)
        if response.status_code == 200:
            print("✅ Pushover sent!")
            return {"status": "success"}
        return {"status": "error"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

## Core Functions

In [ ]:
async def get_questions_async(topic: str) -> str:
    try:
        with trace("Questions"):
            result = await Runner.run(clarifying_agent, f"Ask 3 clarifying questions about: {topic}")
            questions = result.final_output
            if not questions.startswith("###"):
                questions = "### Clarifying Questions\n\n" + questions
            return questions
    except Exception as e:
        return f"❌ Error: {str(e)}"

async def run_research_async(topic: str, questions: str, answers: str) -> str:
    try:
        print("\n📊 Evaluating...")
        with trace("Eval"):
            eval_result = await Runner.run(evaluator_agent, f"Topic: {topic}\nQ: {questions}\nA: {answers}")
        
        if not eval_result.final_output.is_satisfactory:
            return f"❌ Need better answers:\n\n{eval_result.final_output.feedback}"
        
        print("✅ Good - researching...\n")
        query = f"{topic}\nQ: {questions}\nA: {answers}"
        
        print("📋 Planning...")
        with trace("Planning"):
            plan_result = await Runner.run(planner_agent, f"Query: {query}")
            plan = plan_result.final_output
        
        print(f"🔍 Running {len(plan.searches)} searches...")
        search_tasks = [Runner.run(search_agent, f"Search: {item.query}\nWhy: {item.reason}") for item in plan.searches]
        search_results = await asyncio.gather(*search_tasks)
        search_texts = [r.final_output for r in search_results]
        
        print("✍️ Writing report...")
        with trace("Writing"):
            report_result = await Runner.run(writer_agent, f"Query: {query}\nResults: {search_texts}")
            report = report_result.final_output.markdown_report
        
        send_notification("🔬 Research Complete!", f"Your research on '{topic[:50]}...' is ready!")
        
        print("✅ Done!")
        return report
        
    except Exception as e:
        return f"❌ Error: {str(e)}"

## Gradio Wrappers

In [ ]:
def get_questions(topic: str) -> str:
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(get_questions_async(topic))

def run_research(topic: str, questions: str, answers: str) -> str:
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_research_async(topic, questions, answers))

## Gradio UI

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔬 Deep Research with Clarifying Questions")
    gr.Markdown("### Samuel Kalu, Team Euclid, Week 2")
    
    with gr.Row():
        with gr.Column(scale=1):
            topic = gr.Textbox(label="Research Topic", placeholder="e.g., Best AI frameworks 2025", lines=2)
            get_q_btn = gr.Button("🤔 Get Questions", variant="primary")
            questions = gr.Markdown(label="📋 Questions", visible=False, height=200)
            answers = gr.Textbox(label="✏️ Your Answers", visible=False, lines=6)
            research_btn = gr.Button("🚀 Research", visible=False, variant="primary")
        
        with gr.Column(scale=1):
            report = gr.Markdown(label="📄 Report", visible=False, height=700, show_copy_button=True)
    
    get_q_btn.click(fn=get_questions, inputs=topic, outputs=questions).then(
        fn=lambda: (gr.update(visible=True), gr.update(visible=True), gr.update(visible=True)),
        outputs=[questions, answers, research_btn]
    )
    
    research_btn.click(fn=run_research, inputs=[topic, questions, answers], outputs=report).then(
        fn=lambda: gr.update(visible=True), outputs=report
    )
    
    gr.Markdown("---\n**Samuel Kalu, Team Euclid, Week 2**")

demo.launch(share=True)